<center><font color='steelblue'> <font size = "6">CORPORACIÓN UNIVERSITARIA MINUTO DE DIOS </font></center><br>
<center><font color='steelblue'> <font size = "5,5">ESCUELA DE INGENIERÍA</font></center><br>
<center><font color='steelblue'> <font size = "5">PROGRAMA INGENIERÍA DE SISTEMAS - Modalidad Presencial</font></center><br>
<center><font color='steelblue'> <font size = "4">BASES DE DATOS MASIVAS</font></center><br>
<center><font color="yellow" size = "4" face = "small fonts">Proyecto Segundo Corte – Arquitectura distribuida, ETL y Mojo</font></center>

**Modalidad:** Aprendizaje Basado en Proyectos (ABP)  
**Entregable:** Un único archivo `ApellidoNombre_ProyectoBDM.ipynb`

> **Propósito:** construir y demostrar una solución distribuida que integre MongoDB, Docker, Mojo (Modular) y una base de datos relacional mediante un proceso ETL reproducible.

Este cuaderno es la guía oficial de desarrollo. El equipo debe completar las celdas marcadas como `TODO`, documentar las decisiones técnicas y conservar las evidencias de ejecución.

## 0. Datos del equipo

Complete esta información antes de comenzar.

- **NRC:** `83362`
- **Asignatura:** Bases de Datos Masivas
- **Docente:** `ALONSO GUEVARA PEREZ`
- **Integrantes:** `ANDERSON ARLEY DAZA DIAZ // LUIS ALEJANDRO JIMENEZ ACERO // DIEGO HERNANDO BARRERA BETANCOURT`
- **Fecha:** `14/10/2026`
- **Sistema operativo ANFITRION:** `Microsoft Windows 11 Home Single Language Versión del sistema operativo: 10.0.26200 N/D Compilación 26200`
- **Sistema operativo DENTRO DE LOS CONTENEDORES DOCKER:** ``
- **Versión Docker:** `Docker version 29.7.2, build a7dcaa6`
- **Versión Mojo:** `TODO`
- **Versión Python:** `Python 3.14.4`

### Integridad académica

El código debe poder ser explicado por todos los integrantes. La inteligencia artificial puede utilizarse como apoyo para comprender, depurar o consultar documentación, pero no sustituye la comprensión.

**Regla:** aportes generados exclusivamente por IA, sin comprensión, adaptación, validación y explicación del estudiante, podrán recibir **0.0** en el componente correspondiente.

# 1. Contexto y reto

La organización dispone de tres fuentes JSON:

- `personas.json`
- `articulos.json`
- `ventas.json`

La información deberá distribuirse en tres nodos MongoDB independientes, procesarse mediante Mojo y finalmente integrarse en un modelo relacional.

### Reto

> ¿Cómo diseñar e implementar una arquitectura distribuida que permita extraer, validar, transformar e integrar datos heterogéneos, garantizando integridad referencial y produciendo información útil para el análisis?

### Condición fundamental

El lenguaje utilizado para la **lógica central de limpieza, validación y transformación** debe ser **Mojo de Modular**.

**No se acepta Mojolicious/Perl como sustituto de Mojo.**

Python puede utilizarse para orquestación desde Jupyter, automatización, visualización y apoyo al proceso, siempre que el equipo diferencie claramente qué responsabilidad corresponde a Python y cuál a Mojo.

# 2. Arquitectura objetivo

```text
                    ┌─────────────────────┐
                    │   ARCHIVOS JSON     │
                    │ personas/articulos  │
                    │ ventas              │
                    └──────────┬──────────┘
                               │
                               ▼
                    ┌─────────────────────┐
                    │   RED DOCKER        │
                    └──────────┬──────────┘
                               │
            ┌──────────────────┼──────────────────┐
            │                  │                  │
            ▼                  ▼                  ▼
      ┌───────────┐      ┌───────────┐      ┌───────────┐
      │ Mongo 1   │      │ Mongo 2   │      │ Mongo 3   │
      │ PERSONAS  │      │ ARTICULOS │      │ VENTAS    │
      └─────┬─────┘      └─────┬─────┘      └─────┬─────┘
            │                  │                  │
            └──────────────────┼──────────────────┘
                               ▼
                    ┌─────────────────────┐
                    │       MOJO          │
                    │ Extract / Validate  │
                    │ Transform / Integrate│
                    └──────────┬──────────┘
                               ▼
                    ┌─────────────────────┐
                    │ BD RELACIONAL       │
                    │ PERSONAS/ARTICULOS  │
                    │ VENTAS              │
                    └──────────┬──────────┘
                               ▼
                    ┌─────────────────────┐
                    │ SQL + VISUALIZACIÓN │
                    └─────────────────────┘
```

### Actividad

Explique:

1. ¿Por qué existen tres nodos MongoDB?
2. ¿Qué ventaja ofrece Docker?
3. ¿Qué responsabilidad tiene Mojo?
4. ¿Por qué se necesita un modelo relacional al final?
5. ¿Dónde se garantiza la integridad referencial?

El equipo debe reproducir o mejorar este diagrama.

# 3. Modelo relacional de referencia

El equipo debe construir el DDL dentro del notebook. **No se entrega una base SQLite ya poblada.**

# 4. Preparación del entorno

El notebook debe crear una estructura reproducible:

```text
proyecto/
├── data/
│   ├── personas.json
│   ├── articulos.json
│   └── ventas.json
├── mojo/
│   ├── src/
│   └── tests/
├── docker/
├── sql/
└── resultados/
```

No utilice rutas absolutas del equipo. Use rutas relativas o dinámicas.

In [ ]:
from pathlib import Path
import os
import sys
import json
import re
import sqlite3
import subprocess
import platform

BASE = Path.cwd()
DATA = BASE / "data"
MOJO_DIR = BASE / "mojo"
DOCKER_DIR = BASE / "docker"
SQL_DIR = BASE / "sql"
RESULTADOS = BASE / "resultados"

for p in [DATA, MOJO_DIR, DOCKER_DIR, SQL_DIR, RESULTADOS]:
    p.mkdir(parents=True, exist_ok=True)

print("Directorio:", BASE)
print("Sistema operativo:", platform.platform())
print("Python:", sys.version.split()[0])

## 4.1 Verificación de archivos

Los tres JSON deben estar disponibles en `data/`. Si el notebook se encuentra en otro directorio, adapte la ruta de manera dinámica.

In [ ]:
required = ["personas.json", "articulos.json", "ventas.json"]

for name in required:
    path = DATA / name
    print(f"{name:20} ->", "OK" if path.exists() else "FALTA", path)

# 5. Perfilamiento inicial de los datos

Antes de transformar información, determine qué contiene realmente cada fuente.

Para cada JSON calcule:

- Número de registros.
- Campos disponibles.
- Campos faltantes.
- Tipos observados.
- Valores potencialmente inválidos.
- Duplicados de las claves.
- Ejemplos de registros problemáticos.

**No modifique los archivos originales.**

In [ ]:
def cargar_json(nombre):
    path = DATA / nombre
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

personas = cargar_json("personas.json")
articulos = cargar_json("articulos.json")
ventas = cargar_json("ventas.json")

print("Personas :", len(personas))
print("Articulos:", len(articulos))
print("Ventas   :", len(ventas))

In [ ]:
# TODO: completar el perfilador.

def perfilar_registros(registros, nombre):
    print(f"\n--- {nombre} ---")
    # TODO: campos, tipos, nulos y ejemplos
    pass

perfilar_registros(personas, "PERSONAS")
perfilar_registros(articulos, "ARTICULOS")
perfilar_registros(ventas, "VENTAS")

## 5.1 Análisis de calidad

Investigue especialmente:

### PERSONAS
- `numeroDocumento`
- `telefono`

### ARTICULOS
- `idArticulo`
- `cantidadArticulo`

### VENTAS
- `idComprador`
- `idArticulo`
- `cantidadProductos`
- `precioTotal`

Defina una regla de negocio y justifíquela para cada tipo de inconsistencia.

**No elimine registros sin explicar la decisión.**

In [ ]:
NUMERIC_RE = re.compile(r"^\d+(?:\.\d+)?$")

def es_numerico(valor):
    # TODO
    return False

def contar_invalidos(registros, campo):
    # TODO
    return None

# Ejemplos de uso:
# print(contar_invalidos(personas, "numeroDocumento"))
# print(contar_invalidos(articulos, "idArticulo"))
# print(contar_invalidos(ventas, "idComprador"))

# 6. Docker: diseño de la infraestructura

El equipo debe construir la infraestructura, no copiar una configuración terminada.

### Servicios mínimos

| Servicio | Responsabilidad |
|---|---|
| `mongo_personas` | documentos de PERSONAS |
| `mongo_articulos` | documentos de ARTICULOS |
| `mongo_ventas` | documentos de VENTAS |
| `relacional` | almacenamiento final relacional |

El componente Mojo puede ejecutarse como proceso del entorno de trabajo o dentro de un contenedor adicional, siempre que quede claramente documentado.

### Requisito

El `compose.yaml` o `docker-compose.yml` debe ser **generado desde el notebook** para que el único entregable sea autosuficiente.

## 6.1 Variables de infraestructura

Complete:

- **Red Docker:** `TODO`
- **Mongo PERSONAS:** `TODO`
- **Mongo ARTICULOS:** `TODO`
- **Mongo VENTAS:** `TODO`
- **Base relacional:** `TODO`
- **Servicio Mojo:** `TODO`

Justifique puertos, nombres y dependencias.

In [ ]:
compose_path = DOCKER_DIR / "docker-compose.yml"

compose_content = """
# TODO: completar por el equipo
"""

# Cuando esté completo:
# compose_path.write_text(compose_content, encoding="utf-8")
print("Ruta prevista:", compose_path)

## 6.2 Levantamiento de contenedores

Conserve evidencia de:

- `docker compose config`
- `docker compose up -d`
- `docker compose ps`

Debe demostrarse que los servicios están activos y pertenecen a la misma red.

In [ ]:
# TODO: descomentar y completar cuando el compose esté terminado.

# subprocess.run(["docker", "compose", "-f", str(compose_path), "config"], check=True)
# subprocess.run(["docker", "compose", "-f", str(compose_path), "up", "-d"], check=True)
# subprocess.run(["docker", "compose", "-f", str(compose_path), "ps"], check=True)

# 7. Carga de los JSON en MongoDB

Cada fuente debe almacenarse en su propio nodo:

- PERSONAS → `mongo_personas`
- ARTICULOS → `mongo_articulos`
- VENTAS → `mongo_ventas`

Debe demostrarse:

1. Carga.
2. Conteo.
3. Consulta de documentos.
4. Estructura.
5. Independencia de cada nodo.

**No utilice una única instancia MongoDB para simular tres servidores.**

In [ ]:
# TODO: definir y ejecutar los comandos de carga.
# Documente:
# - contenedor
# - base de datos
# - colección
# - comando utilizado

## 7.1 Evidencia de distribución

Responda con evidencia:

- ¿Cuántos documentos quedaron en PERSONAS?
- ¿Cuántos en ARTICULOS?
- ¿Cuántos en VENTAS?
- ¿Cómo comprobó que cada colección está en su nodo correspondiente?

In [ ]:
# TODO: ejecutar consultas de conteo en cada MongoDB.

# 8. Mojo: lenguaje y organización modular

Mojo debe utilizarse para la lógica central del procesamiento.

La CLI de Mojo permite ejecutar y compilar programas Mojo, por ejemplo con `mojo archivo.mojo`, y dispone de comandos como `mojo run` y `mojo build`.

Documentación oficial:
- https://docs.modular.com/mojo/
- https://mojolang.org/docs/cli/

### Requisito

Crear como mínimo una estructura equivalente a:

```text
mojo/
└── src/
    ├── main.mojo
    ├── limpieza.mojo
    ├── validacion.mojo
    └── etl.mojo
```

El nombre exacto puede cambiar si existe una justificación técnica.

## 8.1 Contrato funcional obligatorio

La solución debe cubrir como mínimo:

### Limpieza
`limpiar_valor_numerico`

### PERSONAS
`validar_persona`

### ARTICULOS
`validar_articulo`

### VENTAS
`validar_venta`

### Integridad
`validar_integridad_referencial`

### ETL
`ejecutar_etl`

Cada función debe documentar:

- Propósito.
- Entradas.
- Salidas.
- Regla de validación.
- Manejo de errores.
- Evidencia de ejecución.

In [ ]:
(MOJO_DIR / "src").mkdir(parents=True, exist_ok=True)
(MOJO_DIR / "tests").mkdir(parents=True, exist_ok=True)

print("Estructura Mojo preparada:", MOJO_DIR)

## 8.2 Pruebas unitarias de Mojo

Cada función central debe tener al menos tres escenarios:

- Dato válido.
- Dato inválido.
- Dato vacío/nulo o no recuperable.

El equipo debe demostrar que las funciones se ejecutan realmente con Mojo y no solamente están escritas.

In [ ]:
# TODO: ejecutar desde Jupyter los archivos Mojo de prueba.
# Ejemplo conceptual:
# subprocess.run(["mojo", str(MOJO_DIR / "tests" / "test_validaciones.mojo")], check=True)

# 9. ETL

Implemente:

```text
EXTRACT
   ↓
PROFILE
   ↓
VALIDATE
   ↓
TRANSFORM
   ↓
REFERENTIAL CHECK
   ↓
LOAD
   ↓
VERIFY
```

### Extract
Obtener datos de los tres nodos MongoDB.

### Transform
Aplicar las reglas definidas en Mojo.

### Load
Insertar solamente registros que cumplan el modelo relacional.

### Verify
Comprobar cantidades, tipos, claves y relaciones.

## 9.1 Reglas de transformación

| Fuente | Campo | Regla |
|---|---|---|
| PERSONAS | numeroDocumento | TODO |
| PERSONAS | telefono | TODO |
| ARTICULOS | idArticulo | TODO |
| ARTICULOS | cantidadArticulo | TODO |
| VENTAS | idComprador | TODO |
| VENTAS | idArticulo | TODO |
| VENTAS | cantidadProductos | TODO |
| VENTAS | precioTotal | TODO |

### Pregunta crítica

¿Qué hace su ETL cuando un valor no puede transformarse de forma confiable?

Explique y justifique.

# 10. Base de datos relacional

Construya la base relacional dentro del notebook.

Se recomienda SQLite para facilitar la reproducibilidad.

### Requisitos

- PK en PERSONAS.
- PK en ARTICULOS.
- PK en VENTAS.
- FK `VENTAS.idComprador`.
- FK `VENTAS.idArticulo`.
- Tipos compatibles.
- Integridad referencial activada.

**El notebook debe crear la base desde cero.**

In [ ]:
db_path = RESULTADOS / "almacen.sqlite"

conexion = sqlite3.connect(db_path)
conexion.execute("PRAGMA foreign_keys = ON")

# TODO: crear las tablas mediante DDL escrito por el equipo.
# No utilizar una base de datos ya poblada.

conexion.close()

print("Base relacional:", db_path)

# 11. Carga final y verificación

Después del ETL:

1. Consultar PERSONAS.
2. Consultar ARTICULOS.
3. Consultar VENTAS.
4. Contar registros.
5. Identificar duplicados.
6. Comprobar FK.
7. Comprobar registros huérfanos.
8. Comparar con el perfilamiento inicial.

La pérdida de registros debe poder explicarse.

In [ ]:
# TODO: consultar conteos por tabla y mostrar muestras.

# conexion = sqlite3.connect(db_path)
# conexion.execute("PRAGMA foreign_keys = ON")
# ...

# 12. Pruebas de integridad referencial

Realice como mínimo:

### Prueba A
Ventas cuyo comprador no existe.

### Prueba B
Ventas cuyo artículo no existe.

### Prueba C
Duplicidad de claves.

### Prueba D
Valores nulos en campos obligatorios.

### Prueba E
Tipos incompatibles.

Cada prueba debe presentar:

**consulta → resultado → interpretación**

In [ ]:
# TODO: implementar las consultas de integridad.

# 13. Consulta analítica 1 – Cinco artículos más vendidos

Construya una consulta SQL que determine los cinco artículos con mayor cantidad de unidades vendidas.

Debe mostrar:

- Identificador.
- Nombre.
- Cantidad total vendida.

Luego construya una visualización apropiada e interprete el resultado.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

consulta = """
-- TODO: escribir la consulta SQL
"""

# df = pd.read_sql_query(consulta, conexion)
# display(df)
# TODO: gráfico

# 14. Consulta analítica 2 – Cinco compradores con más compras

Determine los cinco compradores con mayor número de compras.

Diferencie claramente:

- número de transacciones
- cantidad de productos

Justifique la métrica utilizada.

In [ ]:
consulta = """
-- TODO
"""

# TODO: ejecutar y graficar

# 15. Consulta analítica 3 – Distribución de precios

Obtenga los cinco primeros artículos según el criterio indicado en el proyecto y analice su distribución de precios.

El gráfico debe tener título, ejes identificados y unidades cuando correspondan.

In [ ]:
consulta = """
-- TODO
"""

# TODO: ejecutar y visualizar

# 16. Consulta analítica 4 – Cinco artículos menos vendidos

Determine los cinco artículos con menor cantidad de unidades vendidas.

### Importante

Defina cómo tratará los artículos que nunca aparecen en VENTAS.

Explique por qué su consulta sí o no los incluye.

In [ ]:
consulta = """
-- TODO
"""

# TODO: ejecutar y visualizar

# 17. Reproducibilidad

Demuestre que el notebook puede reconstruir la solución.

### Prueba

1. Detener los contenedores.
2. Eliminar los contenedores.
3. Eliminar la base relacional generada.
4. Ejecutar nuevamente las celdas necesarias.
5. Comprobar que los resultados se reconstruyen.

No se aceptan rutas personales ni archivos ocultos del equipo.

In [ ]:
print("Python:", sys.version.split()[0])

# TODO: comprobar versiones disponibles.
# subprocess.run(["docker", "--version"], check=False)
# subprocess.run(["docker", "compose", "version"], check=False)
# subprocess.run(["mojo", "--version"], check=False)

# 18. Defensa técnica

Cada integrante debe estar preparado para responder individualmente:

1. ¿Por qué utilizar tres nodos MongoDB?
2. ¿Qué problema de calidad encontró?
3. ¿Qué regla utilizó para solucionarlo?
4. ¿Qué función Mojo realiza la transformación?
5. ¿Cómo se garantiza la integridad referencial?
6. ¿Qué ocurre si una venta apunta a un artículo inexistente?
7. ¿Qué parte del proyecto realiza Python?
8. ¿Qué parte realiza Mojo?
9. ¿Cómo se reconstruye la solución desde cero?
10. ¿Qué decisión técnica cambiaría y por qué?

# 19. Registro de uso de IA

| Herramienta | Uso | Aporte generado | Modificación realizada | Validación |
|---|---|---|---|---|
| TODO | TODO | TODO | TODO | TODO |

### Declaración

El equipo declara que puede explicar y defender el código presentado y que las herramientas de IA, si fueron utilizadas, fueron empleadas como apoyo y no como sustituto del aprendizaje.

# 20. Conclusiones

Escriba mínimo tres conclusiones técnicas.

Deben responder:

1. ¿Qué aprendió sobre arquitectura distribuida?
2. ¿Qué aprendió sobre calidad y ETL?
3. ¿Qué aprendió sobre Mojo?
4. ¿Qué ventajas y limitaciones encontró al integrar MongoDB, Docker, Mojo y SQLite?
5. ¿Qué mejoraría en una segunda versión?

Evite conclusiones genéricas.

# 21. Referencias

Como mínimo consulte documentación oficial:

- Mojo / Modular: https://docs.modular.com/mojo/
- Mojo CLI: https://mojolang.org/docs/cli/
- Docker Documentation: https://docs.docker.com/
- MongoDB Documentation: https://www.mongodb.com/docs/
- SQLite Documentation: https://www.sqlite.org/docs.html

Agregue las fuentes adicionales utilizadas y cite dentro del notebook cuando corresponda.

# 22. Lista de comprobación antes de entregar

- [ ] El notebook tiene los integrantes.
- [ ] No existen rutas absolutas personales.
- [ ] Los tres JSON originales no fueron modificados.
- [ ] Existen tres nodos MongoDB independientes.
- [ ] Docker levanta correctamente la arquitectura.
- [ ] Cada MongoDB contiene su fuente correspondiente.
- [ ] Se realizó perfilamiento de datos.
- [ ] Se documentaron problemas de calidad.
- [ ] Las funciones centrales están implementadas en Mojo.
- [ ] Existen pruebas de las funciones Mojo.
- [ ] El ETL está documentado.
- [ ] Se creó la BD relacional desde cero.
- [ ] Se validaron PK y FK.
- [ ] Se comprobaron registros huérfanos.
- [ ] Se realizaron las cuatro consultas analíticas.
- [ ] Se generaron los gráficos.
- [ ] Cada resultado tiene interpretación.
- [ ] Se registró el uso de IA.
- [ ] El notebook se ejecuta de principio a fin.
- [ ] Las conclusiones son propias del equipo.

# ENTREGABLE ÚNICO

## `ApellidoNombre_ProyectoBDM.ipynb`

El notebook debe contener **código + resultados + evidencias + análisis + conclusiones**.

> **No entregue ZIP. No entregue archivos adicionales. El notebook debe generar los archivos de configuración necesarios para reproducir la solución.**

### Criterio central

> **No se evalúa solamente que la solución funcione; se evalúa que el equipo pueda demostrar, explicar y defender cómo y por qué funciona.**